# Hybrid RAG retrieval benchmark — standalone reproduction

This cleaned notebook uses the packaged, complete 713-row `data/chunks.csv` corpus and 250 labeled queries in `data/expanded_tickets.csv`. No raw policy PDFs or API key are needed for this benchmark. It runs a reproducible BM25 baseline and document-level retrieval evaluation using the customer query and canonical policy filename stem. It does not recreate dense-model or reranker scores without model packages/checkpoints.

The source Colab contained repeated exploratory attempts and errors. This replaces those cells with an ordered, error-free workflow; outputs and execution counters are intentionally cleared.


In [ ]:
from pathlib import Path
import re
import math
from collections import Counter
import numpy as np
import pandas as pd

# Works when opened from this notebooks/ directory or the project root.
if Path("../data/chunks.csv").exists():
    DATA = Path("../data")
elif Path("data/chunks.csv").exists():
    DATA = Path("data")
else:
    raise FileNotFoundError("Expected packaged data/chunks.csv and data/expanded_tickets.csv")

chunks = pd.read_csv(DATA / "chunks.csv").fillna("")
tickets = pd.read_csv(DATA / "expanded_tickets.csv").fillna("")
required_chunks = {"chunk_id", "source_file", "text"}
required_tickets = {"ticket_id", "customer_query", "primary_expected_doc_id"}
assert required_chunks <= set(chunks.columns), required_chunks - set(chunks.columns)
assert required_tickets <= set(tickets.columns), required_tickets - set(tickets.columns)
assert chunks["chunk_id"].is_unique
assert tickets["ticket_id"].is_unique

def canonical_doc(filename):
    return re.sub(r"\.pdf$", "", str(filename), flags=re.I)

chunks["canonical_doc_id"] = chunks["source_file"].map(canonical_doc)
missing = sorted(set(tickets["primary_expected_doc_id"]) - set(chunks["canonical_doc_id"]))
assert not missing, f"Unmatched ticket target IDs: {missing[:5]}"
print(f"Loaded {len(chunks)} chunks across {chunks.canonical_doc_id.nunique()} policies and {len(tickets)} tickets.")


## BM25 ranking

The compact implementation below uses whitespace-independent tokenization, BM25 with `k1=1.5`, `b=0.75`, and the full context-enriched chunk `text`. Chunk scores are aggregated to each policy by maximum chunk score; the per-ticket document ranking is then deduplicated. This is a runnable baseline, not a claim of exact parity with every historical notebook run.


In [ ]:
TOKEN_RE = re.compile(r"[a-z0-9]+(?:['’-][a-z0-9]+)*", re.I)
def tokenize(text):
    return TOKEN_RE.findall(str(text).lower())

corpus_tokens = [tokenize(text) for text in chunks["text"]]
doc_freq = Counter(token for tokens in corpus_tokens for token in set(tokens))
n_docs = len(corpus_tokens)
avgdl = sum(map(len, corpus_tokens)) / max(n_docs, 1)
idf = {term: math.log(1 + (n_docs - freq + 0.5) / (freq + 0.5))
       for term, freq in doc_freq.items()}

def bm25_scores(query, k1=1.5, b=0.75):
    qterms = tokenize(query)
    scores = np.zeros(n_docs, dtype=float)
    for i, tokens in enumerate(corpus_tokens):
        counts = Counter(tokens); dl = len(tokens)
        for term in qterms:
            tf = counts.get(term, 0)
            if tf:
                scores[i] += idf.get(term, 0.0) * tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl / max(avgdl, 1e-12)))
    return scores

def ranked_documents(query):
    scores = bm25_scores(query)
    frame = pd.DataFrame({"doc": chunks["canonical_doc_id"], "score": scores})
    # max chunk score per document; stable tie break by canonical ID
    return (frame.groupby("doc", as_index=False)["score"].max()
            .sort_values(["score", "doc"], ascending=[False, True], kind="mergesort")["doc"].tolist())

print("BM25 index ready:", n_docs, "chunk documents; mean token length", round(avgdl, 1))


## Document-level evaluation

Recall@K is the fraction of tickets whose primary expected policy appears in the first K unique policy documents. MRR@10 is the mean reciprocal rank of that policy when found in the first ten. Secondary labels are not included in this primary-target metric.


In [ ]:
rankings = [ranked_documents(q) for q in tickets["customer_query"]]
truth = tickets["primary_expected_doc_id"].tolist()

def document_metrics(rankings, truth, ks=(1, 3, 5, 10)):
    assert len(rankings) == len(truth)
    result = {f"Recall@{k}": 0 for k in ks}
    reciprocal = []
    for ranking, target in zip(rankings, truth):
        rank = ranking.index(target) + 1 if target in ranking else None
        for k in ks:
            result[f"Recall@{k}"] += int(rank is not None and rank <= k)
        reciprocal.append(1 / rank if rank is not None and rank <= max(ks) else 0.0)
    n = max(len(truth), 1)
    result = {k: v / n for k, v in result.items()}
    result[f"MRR@{max(ks)}"] = float(np.mean(reciprocal)) if reciprocal else 0.0
    return result

results = document_metrics(rankings, truth)
results_df = pd.DataFrame([{"Pipeline": "BM25 on packaged chunks", **results}])
display(results_df.style.format({c: "{:.3f}" for c in results_df.columns if c != "Pipeline"}))


## Reproduction boundaries and security

The full chunk text and labeled tickets are included, so sparse retrieval can run without the 100 source PDFs. `src/phase1_ingest.py` documents/reimplements PDF extraction for users who separately have the originals. Dense embeddings, cross-encoder checkpoints, and historical per-query score arrays are not included; rerunning those stages requires model downloads and compute. Never store API credentials in notebook source or outputs. A credential was exposed as a visible interactive input in the uploaded source notebook; rotate/revoke it if it is real.
